# Building Connectivity Matrices

A **structural connectivity matrix** (connectome) summarises how strongly each pair of brain regions is connected by white matter streamlines. It is the core output of a tractography-based connectomics study.

## The pipeline

```
Whole-brain tractogram (.tck)
        +
Parcellation atlas (e.g. Desikan-Killiany)
        ↓
Count streamlines connecting each pair of regions
        ↓
N×N connectivity matrix  (N = number of parcels)
```

## Weighting options

Raw streamline counts are **biased**: long tracts accumulate more points than short tracts; large regions intercept more streamlines than small ones. SIFT2 weights correct for FOD-based biases; additional normalisation is needed for region size.

| Weight | Formula | Effect |
|---|---|---|
| Raw count | $w_i = 1$ | Biased by length + region size |
| SIFT2 weighted | $w_i = \text{SIFT2 weight}$ | Corrects FOD bias |
| Length-weighted | $w_i = L_i$ | Emphasises long connections |
| FA-weighted | $w_i = \bar{FA}_i$ | Proxy for fibre integrity |
| Normalised | $C_{ij} / (A_i + A_j)$ | Corrects for region size |

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import run

data_dir  = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir  = Path('../../data/hcp/100307/preprocessed')
csd_dir   = Path('../../data/hcp/100307/csd')
tck_dir   = Path('../../data/hcp/100307/tractography')
conn_dir  = Path('../../data/hcp/100307/connectome')
conn_dir.mkdir(parents=True, exist_ok=True)

# Tractogram (use the 1M iFOD2 from previous chapter)
tck_file     = str(tck_dir / 'prob_iFOD2_1M.tck')
sift_weights = str(tck_dir / 'sift2_weights.txt')

# Parcellation (HCP Atlas_ROIs — native space)
parcellation = str(data_dir.parent.parent / 'MNINonLinear/ROIs/Atlas_ROIs.2.nii.gz')

print('Setup complete.')

## Step 1: Streamline filtering with SIFT2

SIFT2 assigns a weight to each streamline so the total weighted density matches the FOD integral. This corrects for tractography biases.

In [ ]:
# ─── [MRtrix3] tcksift2 ───────────────────────────────────────────────────────
wm_fod_norm = str(csd_dir / 'wmfod_norm.mif')

sift2_cmd = [
    'tcksift2',
    tck_file,
    wm_fod_norm,
    sift_weights,
    '-out_mu', str(conn_dir / 'sift2_mu.txt'),
    '-force',
]
print('[MRtrix3] tcksift2 command:')
print(' '.join(sift2_cmd))

# result = subprocess.run(sift2_cmd, capture_output=True, text=True)
# print(result.stdout or '(done)')
print()
print('Runtime: ~10-30 min for 1M streamlines')
print('>> Uncomment to run')

## Step 2A: Build connectivity matrix with MRtrix3

In [ ]:
# ─── [MRtrix3] tck2connectome ─────────────────────────────────────────────────
#
# Counts streamlines connecting each pair of parcels.
# -symmetric : C_ij = C_ji (undirected connectivity)
# -zero_diagonal : set diagonal to 0 (no self-connections)
# -tck_weights_in : use SIFT2 weights for weighted count

conn_raw    = str(conn_dir / 'connectome_raw.csv')
conn_sift2  = str(conn_dir / 'connectome_sift2.csv')
conn_fa     = str(conn_dir / 'connectome_fa.csv')

# Raw streamline count
raw_cmd = [
    'tck2connectome',
    tck_file,
    parcellation,
    conn_raw,
    '-symmetric', '-zero_diagonal',
    '-force',
]
print('[MRtrix3] tck2connectome (raw count):')
print(' '.join(raw_cmd))
print()

# SIFT2-weighted count
sift2_cmd = [
    'tck2connectome',
    tck_file,
    parcellation,
    conn_sift2,
    '-tck_weights_in', sift_weights,
    '-symmetric', '-zero_diagonal',
    '-force',
]
print('[MRtrix3] tck2connectome (SIFT2-weighted):')
print(' '.join(sift2_cmd))
print()

# FA-weighted (mean FA along each streamline)
fa_map = str(Path('../../data/hcp/100307/dti') / 'fsl_dti_FA.nii.gz')
fa_cmd = [
    'tck2connectome',
    tck_file,
    parcellation,
    conn_fa,
    '-scale_invnodevol',        # normalise by region size
    '-stat_edge', 'mean',
    '-image', fa_map,           # mean FA along each streamline as edge weight
    '-symmetric', '-zero_diagonal',
    '-force',
]
print('[MRtrix3] tck2connectome (FA-weighted, size-normalised):')
print(' '.join(fa_cmd))
print()
print('>> Uncomment to run after generating tck_file and sift_weights')

# for cmd in [raw_cmd, sift2_cmd, fa_cmd]:
#     result = subprocess.run(cmd, capture_output=True, text=True)
#     print(result.stdout or '(done)')

## Step 2B: Build connectivity matrix with DIPY

In [ ]:
# ─── [DIPY] connectivity_matrix ───────────────────────────────────────────────
from dipy.tracking.utils import connectivity_matrix
from dipy.io.streamline import load_trk

# Load streamlines
dipy_trk = str(tck_dir / 'prob_dipy.trk')

if Path(dipy_trk).exists():
    img  = nib.load(str(data_dir / 'data.nii.gz'))
    sft  = load_trk(dipy_trk, img)
    sls  = sft.streamlines

    # Load parcellation in DWI space (requires registration — use FSL FLIRT)
    # For demo, we use the HCP atlas directly if already in diffusion space
    if Path(parcellation).exists():
        parc_img  = nib.load(parcellation)
        parc_data = parc_img.get_fdata().astype(int)

        # Build connectivity matrix
        M, grouping = connectivity_matrix(
            sls,
            affine=img.affine,
            label_volume=parc_data,
            return_mapping=True,
            mapping_as_streamlines=False,
        )
        # Remove background (label 0)
        M = M[1:, 1:]
        # Make symmetric
        M = M + M.T
        np.fill_diagonal(M, 0)

        np.savetxt(str(conn_dir / 'connectome_dipy.csv'), M, delimiter=',')
        print(f'[DIPY] Connectivity matrix shape: {M.shape}')
        print(f'  Non-zero connections: {(M > 0).sum() // 2}')
        print(f'  Max connection weight: {M.max():.0f}')
    else:
        print('Parcellation not found — download HCP Atlas_ROIs or register your own')
else:
    print('Run DIPY probabilistic tractography first')

## Visualise the connectome

In [ ]:
# Load and plot connectivity matrices
import matplotlib.colors as mcolors

matrices = {}
for label, path in [
    ('Raw count (MRtrix3)', conn_raw),
    ('SIFT2-weighted (MRtrix3)', conn_sift2),
    ('DIPY', str(conn_dir / 'connectome_dipy.csv')),
]:
    if Path(path).exists():
        matrices[label] = np.loadtxt(path, delimiter=',')

if matrices:
    n = len(matrices)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (label, M) in zip(axes, matrices.items()):
        # Log-scale for display (raw counts span orders of magnitude)
        im = ax.imshow(np.log1p(M), cmap='hot_r', aspect='auto')
        ax.set_title(label, fontsize=10)
        ax.set_xlabel('Region index')
        ax.set_ylabel('Region index')
        plt.colorbar(im, ax=ax, label='log(count + 1)')

    fig.suptitle('Structural Connectivity Matrices', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    # Demonstrate with a simulated matrix
    print('No real matrices found — showing a simulated example')
    rng = np.random.default_rng(42)
    M_sim = np.zeros((84, 84))
    idx   = np.triu_indices(84, k=1)
    M_sim[idx] = rng.exponential(scale=500, size=len(idx[0])) * (
        rng.random(len(idx[0])) > 0.4)  # 60% of pairs have some connection
    M_sim = M_sim + M_sim.T

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(np.log1p(M_sim), cmap='hot_r')
    ax.set_title('Simulated 84-region connectome (Desikan-Killiany atlas)\n'
                 'log scale — run tractography for real data', fontsize=11)
    ax.set_xlabel('Region')
    ax.set_ylabel('Region')
    plt.colorbar(im, label='log(count + 1)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Graph-theoretic measures using NetworkX
import networkx as nx

if matrices:
    M = list(matrices.values())[0]
elif 'M_sim' in dir():
    M = M_sim
else:
    M = None

if M is not None:
    # Threshold: keep top 10% of connections
    thr = np.percentile(M[M > 0], 90)
    M_thr = M.copy()
    M_thr[M_thr < thr] = 0

    G = nx.from_numpy_array(M_thr)

    print('=== Graph-theoretic summary ===')
    print(f'  Nodes (regions)   : {G.number_of_nodes()}')
    print(f'  Edges (connections): {G.number_of_edges()}')
    print(f'  Density           : {nx.density(G):.3f}')
    if nx.is_connected(G):
        print(f'  Avg shortest path : {nx.average_shortest_path_length(G):.2f}')
        print(f'  Global efficiency : {nx.global_efficiency(G):.3f}')
    print(f'  Avg clustering    : {nx.average_clustering(G):.3f}')

    # Degree distribution
    degrees = [d for _, d in G.degree()]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(degrees, bins=20, color='steelblue', edgecolor='white')
    ax.set_xlabel('Node degree (# connections)')
    ax.set_ylabel('Count')
    ax.set_title('Degree distribution of the structural connectome\n'
                 '(top 10% thresholded)')
    plt.tight_layout()
    plt.show()

---

## Summary: MRtrix3 vs DIPY for connectomics

| | MRtrix3 tck2connectome | DIPY connectivity_matrix |
|---|---|---|
| SIFT2 weights | Native support | Manual (apply weights) |
| Edge weight options | Length, FA, volume, custom | Custom (numpy operations) |
| Speed | Very fast | Moderate |
| Flexibility | CLI flags | Full Python control |
| Graph analysis | External (Python, MATLAB) | NetworkX, scipy |

**FSL does not have a native connectivity matrix tool** — this is handled by MRtrix3 or DIPY.

**Next**: [Reproducibility Module →](../05_reproducibility/00_overview.ipynb)